# Synthetic Chest X-rays with Stable Diffusion 3

Generates synthetic chest X-rays from the Indiana University radiology reports:
OpenAI `gpt-4.1-mini` extracts a structured clinical description from each report,
and Stable Diffusion 3 Medium renders it locally on the GPU.

Supports both modes — `txt2img` (purely synthetic) and `img2img` (conditioned on
the patient's real X-ray) — and is **checkpointed**: metadata is saved after
every image, `--resume` skips uids already done, and progress can be published
as a Kaggle Dataset so it survives across sessions, not just within one. On a
"GPU T4 x2" session, cell 12c splits a txt2img run across both GPUs at once.

**This copy runs txt2img only** — cells 12b/13/14/15 (img2img run, compare,
strength sweep, local zip export) are commented out. Cell 7 is pre-filled to
resume from an existing checkpoint (`whiteflags26/synthetic-xrays-sd3`, 65
reports already generated) rather than starting fresh.

## Before you run

**Kaggle settings** (right sidebar → Session options):
- **Accelerator:** GPU L4 x4 preferred. GPU T4 x2 works too — use cell 12c to
  run both GPUs in parallel and roughly halve the wall time — but is still
  several times slower per-GPU than an L4.
- **Internet:** ON — required to clone the repo and download the model.

**Kaggle Secrets** (Add-ons → Secrets):
| Secret | Required | Notes |
|---|---|---|
| `HF_TOKEN` | yes | From [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) |
| `OPENAI_API_KEY` | yes | Prompt extraction, ~$0.65 for all 1000 reports |
| `KAGGLE_USERNAME` / `KAGGLE_KEY` | optional | Only for publishing/pulling checkpoints as a Kaggle Dataset (cells 5, 7, 16). From [kaggle.com/settings](https://www.kaggle.com/settings) → Create New Token |

**HuggingFace license:** accept it at
[huggingface.co/stabilityai/stable-diffusion-3-medium](https://huggingface.co/stabilityai/stable-diffusion-3-medium)
with the same account the token belongs to. The repo is gated — downloads fail
with a 401 until you do.

**Dataset:** `raddar/chest-xrays-indiana-university` (Add data) is **not
needed** for this txt2img-only run — it's only used by img2img and
aspect-ratio matching (both disabled here). Cell 8 skips it gracefully either
way.

No GCP service account is needed — this notebook does not touch Google Cloud.

## Checkpointing and persistence — how this actually works

- **Interrupted mid-run?** Metadata is written after every single image, not
  just at the end. Re-run the exact same cell — `--resume` is already in the
  run commands below — and it skips every uid that already has a complete
  result and continues from there. A killed session costs at most the one
  image that was in flight.
- **`/kaggle/working` is not a guaranteed persistent store.** It typically
  survives you closing and reopening *this same notebook draft*, but Kaggle
  does not promise that, and it is wiped by a session "Factory reset". Do not
  rely on it as your only copy of a long run.
- **The durable option is a Kaggle Dataset.** Cell 16 publishes `output/` as a
  dataset you own; cell 7 pulls the latest version back into `output/` at the
  start of a later session, so `--resume` picks up exactly where a *previous
  session* — not just a previous cell — left off. This is also how you turn
  the finished result into something reusable: attach it as input to another
  notebook, or share the link.
- Both the dataset cells are **optional** — skip them and the notebook still
  runs end to end using `/kaggle/working` alone for a single session.

## 1. Verify the GPU

Fail here rather than 40 minutes into a model download.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Set Accelerator to GPU in the session options "
        "(right sidebar) and restart the session. SD3 is unusably slow on CPU."
    )

props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1024**3
CAPABILITY = torch.cuda.get_device_capability(0)

N_GPUS = torch.cuda.device_count()

print(f"GPUs visible: {N_GPUS}")
print(f"GPU:        {props.name}")
print(f"VRAM:       {VRAM_GB:.1f} GB")
print(f"Capability: {CAPABILITY[0]}.{CAPABILITY[1]}")
print(f"torch:      {torch.__version__}")
if N_GPUS >= 2:
    print(f"\n→ {N_GPUS} GPUs detected — cell 12c can run txt2img sharded across both.")

# bfloat16 needs compute capability 8.0+. On a T4 (7.5) it yields black images.
SUPPORTS_BF16 = CAPABILITY[0] >= 8
# SD3 Medium needs ~16-18GB at fp16; below that, offload to CPU.
NEEDS_OFFLOAD = VRAM_GB < 20

print()
print(f"→ dtype:       {'bfloat16' if SUPPORTS_BF16 else 'float16'}")
print(f"→ CPU offload: {NEEDS_OFFLOAD}")
if NEEDS_OFFLOAD:
    print("  (expect ~60-120s per image; an L4 does ~8-15s)")

## 2. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/whiteflags26/synth-dataset-pipeline.git"
REPO_DIR = "/kaggle/working/synth-dataset-pipeline"

if os.path.exists(REPO_DIR):
    print("Repo already present — pulling latest.")
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls

## 3. Install dependencies

Kaggle already ships torch — reinstalling it is a multi-GB no-op that often
breaks the CUDA build, so it is deliberately excluded.

No `langchain-google-*` or `google-cloud-aiplatform` either: this notebook uses
OpenAI for text and SD3 for images, so the Google stack is dead weight.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf safetensors
!pip install -q langchain langchain-core langchain-openai openai
!pip install -q pandas python-dotenv Pillow pydantic tqdm

import importlib, inspect
import diffusers, transformers

print(f"diffusers:    {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")

# The prompt adapter depends on prompt_3 and max_sequence_length existing on
# the pipeline's __call__. Verify against the build that actually installed
# rather than trusting the docs.
from diffusers import StableDiffusion3Pipeline, StableDiffusion3Img2ImgPipeline

sig = inspect.signature(StableDiffusion3Pipeline.__call__).parameters
for param in ("prompt_2", "prompt_3", "max_sequence_length", "negative_prompt_3"):
    print(f"  {'OK ' if param in sig else 'MISSING'} {param}")

i2i = inspect.signature(StableDiffusion3Img2ImgPipeline.__call__).parameters
print(f"  {'OK ' if 'strength' in i2i else 'MISSING'} strength (img2img)")

## 4. HuggingFace cache and login

`HF_HOME` must be set **before** any HuggingFace import. Kaggle's default cache
lives on a small volume and the ~15 GB SD3 download will exhaust it.

In [ ]:
import os

# Must precede any huggingface_hub / diffusers model load.
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

HF_TOKEN = secrets.get_secret("HF_TOKEN")
OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

print(f"HF_HOME: {os.environ['HF_HOME']}")
print("Logged in to HuggingFace.")

!df -h /kaggle/working | tail -1

## 5. Kaggle API credentials (optional)

Only needed for cells 7 and 16 — pulling in or publishing a checkpoint as a
Kaggle Dataset, the durable option described above. Skip this if a single
session is enough; everything else in the notebook works without it.

Get a token at [kaggle.com/settings](https://www.kaggle.com/settings) → API →
"Create New Token" — it downloads a `kaggle.json` with a `username` and a
`key`. Store those two values as the Kaggle Secrets `KAGGLE_USERNAME` and
`KAGGLE_KEY` (never paste them directly into a cell).

In [ ]:
import json
from pathlib import Path

try:
    KAGGLE_USERNAME = secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = secrets.get_secret("KAGGLE_KEY")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    kaggle_json.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
    kaggle_json.chmod(0o600)

    HAS_KAGGLE_API = True
    print(f"Kaggle API credentials written for user '{KAGGLE_USERNAME}'.")
except Exception:
    HAS_KAGGLE_API = False
    print(
        "No KAGGLE_USERNAME/KAGGLE_KEY secrets set — dataset publish/resume "
        "cells (7, 16) will be skipped. This is optional; everything else "
        "in the notebook still runs."
    )

## 6. Write `.env`

`MODE` here is the single switch that drives both the config and the run cells
below. `DROP_T5` is the fix for an out-of-memory error from cell 12c (dual-GPU)
— set it `True` and re-run this cell, then re-run 12c.

In [ ]:
# "txt2img" or "img2img" — drives SD3_MODE and the default run cells.
MODE = "txt2img"

# Set True if cell 12c (dual-GPU) OOMs — either a CUDA OutOfMemoryError in
# gpu0.log/gpu1.log, or a bare crash with no CUDA traceback (host RAM
# exhausted: two parallel shards each hold a full CPU-offloaded copy of the
# model, so dual-GPU roughly doubles RAM use vs. one shard). Dropping T5
# frees ~9-10GB per shard — it's ~half the model — at the cost of the
# long-form clinical prompt; only the 77-token CLIP channels remain.
DROP_T5 = False

DTYPE = "bfloat16" if SUPPORTS_BF16 else "float16"
OFFLOAD = "true" if NEEDS_OFFLOAD else "false"

env_content = f"""
# === API Keys ===
OPENAI_API_KEY={OPENAI_API_KEY}
HUGGINGFACE_TOKEN={HF_TOKEN}

# === Prompt extraction ===
CHAT_MODEL=gpt-4.1-mini

# === Image generation ===
IMAGE_GENERATOR=sd3
SD3_MODEL_ID=stabilityai/stable-diffusion-3-medium-diffusers
SD3_MODE={MODE}
SD3_STEPS=28
SD3_GUIDANCE_SCALE=7.0
SD3_HEIGHT=1024
SD3_WIDTH=1024
SD3_IMG2IMG_STRENGTH=0.75
SD3_DTYPE={DTYPE}
SD3_SEED=1234
SD3_MAX_SEQUENCE_LENGTH=512
SD3_ENABLE_CPU_OFFLOAD={OFFLOAD}
SD3_ENABLE_VAE_SLICING=true
SD3_DROP_T5={"true" if DROP_T5 else "false"}
""".strip()

with open(".env", "w") as f:
    f.write(env_content)

# Echo without the secrets.
for line in env_content.splitlines():
    if line.startswith(("OPENAI_API_KEY", "HUGGINGFACE_TOKEN")):
        key = line.split("=")[0]
        print(f"{key}=<set>")
    else:
        print(line)

## 7. Resume from a previous Kaggle Dataset (optional)

`/kaggle/working` is not guaranteed to survive between sessions (see the intro
above). If you published a checkpoint with cell 16 in an earlier session, pull
it back in here before running — `--resume` in cells 12a/12b then continues
from those uids instead of starting over.

Leave `DATASET_SLUG` as `None` on a fresh run, or if you are not using
cross-session checkpoints.

In [ ]:
import shutil
from pathlib import Path

# Set to "yourusername/synthetic-xrays-sd3" — the slug printed by cell 16 in a
# previous session — to pull that checkpoint in before running.
DATASET_SLUG = "whiteflags26/synthetic-xrays-sd3"

if DATASET_SLUG and HAS_KAGGLE_API:
    try:
        import kagglehub
    except ImportError:
        !pip install -q kagglehub
        import kagglehub

    print(f"Downloading previous output from {DATASET_SLUG} ...")
    downloaded = Path(kagglehub.dataset_download(DATASET_SLUG))
    print(f"   → {downloaded}")

    dest = Path("output")
    dest.mkdir(exist_ok=True)
    # cell 16 publishes with an output/ subfolder inside the dataset; merge
    # its contents in without clobbering anything a fresh clone already wrote.
    src = downloaded / "output" if (downloaded / "output").exists() else downloaded
    copied = 0
    for item in src.rglob("*"):
        if item.is_dir():
            continue
        target = dest / item.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(item, target)
            copied += 1

    n_meta = len(list(dest.glob("metadata*.json"))) + len(list(dest.glob("*/metadata*.json")))
    print(f"Merged {copied} file(s) into ./output — {n_meta} metadata file(s) present.")
    print("Pass --resume to the run cells below to continue from here.")
elif DATASET_SLUG:
    print("DATASET_SLUG is set but Kaggle API credentials are missing — see cell 5.")
else:
    print("DATASET_SLUG not set — starting fresh (nothing to resume across sessions).")

## 8. Check the dataset (optional for txt2img)

The `raddar/chest-xrays-indiana-university` dataset is only needed for
**img2img** mode and for matching each output's aspect ratio to its original
X-ray. Pure **txt2img** works without it — attaching it is optional, and this
cell never fails the notebook if it's missing; it just disables those two
things and continues at a fixed 1024x1024.

`indiana_projections.csv` (the report → filename lookup) is always available
regardless — it's a small CSV already in the repo, not part of the attached
image dataset.

In [ ]:
import os

CANDIDATES = [
    "/kaggle/input/chest-xrays-indiana-university/images/images_normalized",
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized",
]

IMAGES_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
HAS_IMAGES_DIR = IMAGES_DIR is not None

# The CSV lookup itself has nothing to do with whether the image files are
# attached — load it unconditionally so later cells can always rely on it.
from pipeline.report_parser import load_projections
proj = load_projections("indiana_projections.csv")
print(f"Projections: {len(proj)} uids (from indiana_projections.csv)")

if not HAS_IMAGES_DIR:
    print(
        "\nNo images directory found under /kaggle/input — continuing without "
        "it. This is fine for txt2img (this notebook's current MODE); img2img "
        "and aspect-ratio matching need it. Available under /kaggle/input:"
    )
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count("/") - 2
        if depth <= 3:
            print("  " * depth + f"{os.path.basename(root) or root}/  ({len(files)} files)")
    print(
        "\nTo enable img2img: Add data -> search "
        "'raddar/chest-xrays-indiana-university' -> Add, then re-run this cell."
    )
else:
    files = os.listdir(IMAGES_DIR)
    print(f"IMAGES_DIR: {IMAGES_DIR}")
    print(f"Files:      {len(files)}")
    print(f"Sample:     {files[:3]}")

    for p in proj[1]:
        exists = os.path.exists(f"{IMAGES_DIR}/{p.filename}")
        print(f"  {'OK ' if exists else 'MISSING'} {p.filename} ({p.projection})")

## 9. Warm-up — both pipelines

The first cell that can realistically fail: gated-repo 401, out of disk, OOM.
It fails in ~3 minutes instead of 40 and caches the model for the real runs.

It also proves `from_pipe` shares weights — allocated VRAM should barely move
when the img2img pipeline is bound. This part doesn't need the attached
dataset: without it, a synthetic placeholder stands in for the real X-ray so
the code path is still exercised end to end.

The model is **released at the end**. Cells 12a/12b run `python -m pipeline` as
subprocesses that load their own copy, and two copies will not fit on a 16 GB
card.

In [ ]:
import torch, time
from PIL import Image
from pipeline.sd3_generator import load_sd3_pipeline, _load_init_image, _PIPELINE_CACHE

def vram():
    return torch.cuda.memory_allocated() / 1024**3

t0 = time.time()
pipe = load_sd3_pipeline("txt2img")
print(f"\ntxt2img loaded in {time.time() - t0:.0f}s — VRAM allocated: {vram():.2f} GB")

# Comment out img2img pipeline loading - not needed for txt2img only
# before = vram()
# i2i = load_sd3_pipeline("img2img")
# after = vram()
# print(f"img2img bound — VRAM {before:.2f} -> {after:.2f} GB (delta {after - before:.2f} GB)")
# print("Weights are shared." if after - before < 0.5 else "WARNING: weights look duplicated.")

# 4-step throwaway generation to prove txt2img works
img = pipe(prompt="a chest radiograph", num_inference_steps=4,
           guidance_scale=7.0, height=1024, width=1024).images[0]
print(f"\ntxt2img OK: {img.size}")

# Comment out img2img test - not needed for txt2img only
# if HAS_IMAGES_DIR:
#     init = _load_init_image(f"{IMAGES_DIR}/{proj[1][0].filename}", (896, 1088))
# else:
#     # No dataset attached — a flat placeholder still exercises the img2img
#     # call path (dtype, shapes, strength) without needing real data.
#     init = Image.new("RGB", (896, 1088), color=128)
# img2 = i2i(prompt="a chest radiograph", image=init, strength=0.7,
#            num_inference_steps=8, guidance_scale=7.0).images[0]
# print(f"img2img OK: {img2.size}")

# Release before the subprocess runs below.
_PIPELINE_CACHE.clear()
del pipe
# del i2i  # Not needed since i2i is commented out
torch.cuda.empty_cache()
print(f"\nReleased — VRAM allocated: {vram():.2f} GB")

## 10. Prompt smoke test

One report through `gpt-4.1-mini`. Confirms the key works and — more usefully —
shows what actually reaches the CLIP encoders. The CLIP prompt must contain
clinical findings, not just boilerplate.

In [ ]:
from pipeline.report_parser import load_reports
from pipeline.prompt_builder import build_prompt_chain, extract_structured_prompt
from pipeline.image_prompt_formatter import format_image_prompt
from pipeline.sd3_prompt_adapter import split_prompt_for_sd3, build_negative_prompt

records = load_reports("indiana_reports.csv", limit=1, projections=proj)
chain = build_prompt_chain()
structured = extract_structured_prompt(records[0], chain=chain)

print("=== STRUCTURED FIELDS ===")
for k, v in structured.model_dump(exclude={"reference_images", "view"}).items():
    if v:
        print(f"  {k}: {v}")

full = format_image_prompt(structured)
clip_prompt, t5_prompt = split_prompt_for_sd3(full, structured)

print(f"\n=== CLIP PROMPT ({len(clip_prompt.split())} words, budget 60) ===")
print(clip_prompt)
print(f"\n=== T5 PROMPT ({len(t5_prompt.split())} words) ===")
print(t5_prompt[:600] + " ...")
print(f"\n=== NEGATIVE ===")
print(build_negative_prompt(""))
print(f"\nFull prompt was {len(full.split())} words; "
      f"CLIP would have truncated it to ~77 tokens without this split.")

## 11. Checkpoint status

Reads the metadata files directly, so it is safe to re-run any time — before a
run, mid-run in another tab, or after an interruption — to see what is
actually done rather than guessing from the last cell's output.

In [ ]:
import json
from pathlib import Path

def checkpoint_status(subdirs=("txt2img", "img2img")):
    for name in subdirs:
        path = Path(f"output/metadata_{name}.json")
        if not path.exists():
            print(f"{name:8s}  no run yet ({path})")
            continue
        entries = json.loads(path.read_text())
        done = sum(
            1 for e in entries
            if not e.get("error_prompt")
            and e.get("views")
            and all("image_path" in v for v in e["views"])
        )
        failed = len(entries) - done
        status = f"{done} complete"
        if failed:
            status += f", {failed} failed/partial (will retry with --resume)"
        print(f"{name:8s}  {status}  ({len(entries)} total in {path.name})")

checkpoint_status()
print("\nRe-run this cell any time to check progress. The run cells below already")
print("pass --resume, so re-running them after a stop continues rather than restarts.")

## 12a. Run — text to image

Purely synthetic: reference X-rays are ignored. `--output-subdir` keeps this run
separate from the img2img one below, since both write `<uid>.png`.

`--images-dir`/`--projections-csv` are included only when the dataset is
attached (cell 8) — they're used here purely to match each output's aspect
ratio to its original X-ray. Without them txt2img still runs fine, just at a
fixed 1024x1024 square.

`--resume` makes this safe to re-run verbatim after any interruption — it skips
uids already complete in `output/metadata_txt2img.json` and retries only what
failed or never finished. Raise `--limit` for a full run; on Kaggle's 12h cap,
just re-run this same cell if it doesn't finish in one session.

`OFFSET` skips the first N rows of `indiana_reports.csv` before `--limit`
applies — e.g. `OFFSET = 500` starts from row 500. This is a *position* in
the CSV, independent of `--resume` (which skips by uid, based on what's
already in `output/metadata_txt2img.json`). Use it to hand a different
slice of the dataset to another Kaggle session running in parallel, or to
jump ahead without re-scanning uids you already know are done.

In [ ]:
# Skip the first N rows of the CSV before --limit applies. 0 = start from
# the beginning. Independent of --resume — see the note above.

geometry_args = (
    f"--images-dir {IMAGES_DIR} --projections-csv indiana_projections.csv"
    if HAS_IMAGES_DIR else ""
)
if not HAS_IMAGES_DIR:
    print("No images directory — generating at a fixed 1024x1024 (see cell 8).\n")

!python -m pipeline \
    --csv indiana_reports.csv \
    {geometry_args} \
    --generator sd3 \
    --sd3-mode txt2img \
    --sd3-seed 1234 \
    --output-subdir txt2img \
    --resume \
    --offset 0 \
    --limit 4000

## 12b. Run — image to image (disabled — txt2img-only workflow)

**Commented out.** This session only runs `txt2img` — uncomment the cell
below (and re-enable cells 13/14/15, also commented out) if img2img is
needed later.

Same uids, same seed, conditioned on the real X-ray. Requires the dataset
from cell 8 — this cell prints a skip message and does nothing if it isn't
attached, rather than failing.

Also safe to re-run with `--resume` after a stop, same as 12a.

In [ ]:
# if not HAS_IMAGES_DIR:
#     print(
#         "Skipping — img2img needs the attached dataset. In the notebook's "
#         "Add data panel, search 'raddar/chest-xrays-indiana-university' and "
#         "add it, then re-run cell 8 and this cell."
#     )
# else:
#     !python -m pipeline \
#         --csv indiana_reports.csv \
#         --projections-csv indiana_projections.csv \
#         --images-dir {IMAGES_DIR} \
#         --generator sd3 \
#         --sd3-mode img2img \
#         --sd3-strength 0.75 \
#         --sd3-seed 1234 \
#         --output-subdir img2img \
#         --resume \
#         --limit 5

## 12c. Run — 2x GPU parallel (optional, txt2img only)

Alternative to cell 12a when the session has **two GPUs** — Kaggle's "GPU T4
x2" accelerator (Session options → Accelerator). Splits the work in half by
`uid % 2` (see `--num-shards`/`--shard-index` in `pipeline.py`) and launches
one `python -m pipeline` process per GPU, each pinned with
`CUDA_VISIBLE_DEVICES` so it only ever sees its own card — no change to the
SD3 code itself was needed, since `torch.cuda.is_available()` and the default
device always resolve to whichever GPU that process can see.

Roughly halves wall-clock time versus cell 12a on the same accelerator, at the
cost of **~2x system RAM**: with `SD3_ENABLE_CPU_OFFLOAD=true` (the default on
a 16GB card) each process CPU-offloads its own full copy of the model. If you
hit a **host** out-of-memory (not a CUDA OOM — this shows up as the kernel or
a shard process dying with no CUDA error in its log), set `SD3_DROP_T5=true`
in the `.env` cell and re-run, or just use cell 12a instead.

Each shard writes to its own `output/metadata_txt2img_gpu{0,1}.json` and
`output/images/txt2img_gpu{0,1}/`; this cell then runs `merge_shards.py` to
combine them into `output/metadata_txt2img.json` / `output/images/txt2img/` —
the exact files cell 12a would have produced — so cells 13/15/16 work
unchanged regardless of which cell you ran.

`--resume` is passed to both shards, so re-running this cell after a stop only
regenerates what didn't finish — including picking up right where the
existing checkpoint (pulled in by cell 7) left off.

`OFFSET` (same meaning as in cell 12a) skips the first N rows of the CSV
*before* the 2-way shard split — e.g. `OFFSET = 500` hands this GPU pair rows
500 onward instead of the beginning.

In [ ]:
# import os
# import subprocess
# import sys

# # Skip the first N rows of the CSV before the shard split. 0 = start from
# # the beginning. Same meaning as cell 12a's OFFSET.
# OFFSET = 0

# if N_GPUS < 2:
#     print(f"Only {N_GPUS} GPU visible — this cell needs 2. Use cell 12a instead.")
# else:
#     print(f"{N_GPUS} GPUs visible — launching one shard per GPU.\n")

#     geometry_args = (
#         ["--images-dir", IMAGES_DIR, "--projections-csv", "indiana_projections.csv"]
#         if HAS_IMAGES_DIR else []
#     )

#     def launch_shard(gpu_id: int):
#         env = os.environ.copy()
#         env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
#         cmd = [
#             sys.executable, "-m", "pipeline",
#             "--csv", "indiana_reports.csv",
#             *geometry_args,
#             "--generator", "sd3",
#             "--sd3-mode", "txt2img",
#             "--sd3-seed", "1234",
#             "--output-subdir", f"txt2img_gpu{gpu_id}",
#             "--num-shards", "2", "--shard-index", str(gpu_id),
#             "--resume",
#             "--offset", str(OFFSET),
#             "--limit", "200",
#         ]
#         log = open(f"gpu{gpu_id}.log", "w")
#         # Popen returns immediately — both shards are running concurrently,
#         # each on its own GPU, before either .wait() call below blocks.
#         return subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log

#     procs = [launch_shard(i) for i in range(2)]
#     for proc, log in procs:
#         proc.wait()
#         log.close()

#     exit_codes = [proc.returncode for proc, _ in procs]
#     for i, code_ in enumerate(exit_codes):
#         print(f"\n=== gpu{i}.log (tail, exit={code_}) ===")
#         !tail -n 15 gpu{i}.log

#     if any(c != 0 for c in exit_codes):
#         print(
#             "\nAt least one shard exited non-zero — check the log above. "
#             "Not merging; re-run this cell after fixing the issue (--resume "
#             "will skip whatever already completed)."
#         )
#     else:
#         !python merge_shards.py txt2img_gpu0 txt2img_gpu1 --output-subdir txt2img
#         print(
#             "\nMerged into output/metadata_txt2img.json / "
#             "output/images/txt2img/ — same target cell 12a writes to."
#         )

## 13. Compare — original vs txt2img vs img2img (disabled — needs img2img)

**Commented out** — this cell compares txt2img against img2img output, which
needs cell 12b (also disabled). Uncomment both if img2img is turned back on.

The cell that makes the mode choice empirical. Also check `mode` in the caption:
if it reads `txt2img` in the img2img column, that record had no reference image
and fell back.

In [ ]:
# import json, os
# import matplotlib.pyplot as plt
# from PIL import Image

# def load_meta(name):
#     path = f"output/metadata_{name}.json"
#     if not os.path.exists(path):
#         return {}
#     with open(path) as f:
#         return {e["uid"]: e for e in json.load(f)}

# t2i, i2i_meta = load_meta("txt2img"), load_meta("img2img")
# uids = [u for u in t2i if u in i2i_meta]

# if not uids:
#     print("No overlapping uids — run cells 12a and 12b first.")
# else:
#     fig, axes = plt.subplots(len(uids), 3, figsize=(13, 4.4 * len(uids)))
#     if len(uids) == 1:
#         axes = axes.reshape(1, 3)

#     for row, uid in enumerate(uids):
#         entry = t2i[uid]

#         # Original reference X-ray
#         ax = axes[row][0]
#         refs = entry.get("reference_images", [])
#         if refs:
#             ax.imshow(Image.open(f"{IMAGES_DIR}/{refs[0]['filename']}"), cmap="gray")
#             ax.set_title(f"Original (uid={uid})", fontsize=10)
#         else:
#             ax.text(0.5, 0.5, "no reference", ha="center", va="center")
#             ax.set_title(f"Original (uid={uid})", fontsize=10)
#         ax.axis("off")

#         # Generated, one column per mode
#         for col, (label, meta) in enumerate([("txt2img", t2i), ("img2img", i2i_meta)], start=1):
#             ax = axes[row][col]
#             views = meta[uid].get("views", [])
#             view = next((v for v in views if "image_path" in v), None)
#             if view:
#                 ax.imshow(Image.open(view["image_path"]), cmap="gray")
#                 params = view.get("sd3_params", {})
#                 effective = params.get("mode", "?")
#                 extra = f", str={params['strength']}" if "strength" in params else ""
#                 flag = "  <- fell back" if effective != label else ""
#                 ax.set_title(f"{label} (ran: {effective}{extra}){flag}", fontsize=10)
#             else:
#                 err = views[0].get("error_image", "no image") if views else "no views"
#                 ax.text(0.5, 0.5, str(err)[:60], ha="center", va="center", fontsize=8, wrap=True)
#                 ax.set_title(f"{label} — failed", fontsize=10)
#             ax.axis("off")

#     plt.tight_layout()
#     plt.show()

## 14. Strength sweep (img2img tuning) (disabled — needs img2img)

**Commented out** — tunes img2img's `strength`, so it's moot with 12b/13
disabled. Uncomment together with those if img2img comes back.

`strength` is the knob that decides whether img2img output is a synthetic image
or a lightly filtered copy of a real X-ray. Sweep it on one uid and pick the
floor for **this** dataset rather than trusting a default.

Runs in-process, so the model is loaded into the kernel once. Needs an
img2img run (cell 12b) to have completed — skipped otherwise.

In [ ]:
# import torch
# from pathlib import Path
# import matplotlib.pyplot as plt
# from PIL import Image

# from pipeline import config
# from pipeline.sd3_generator import generate_image_sd3

# if not i2i_meta or not uids:
#     print(
#         "Skipping — no img2img run to tune. This needs cell 12b to have "
#         "completed (which needs the dataset from cell 8)."
#     )
# else:
#     STRENGTHS = [0.4, 0.55, 0.7, 0.85, 1.0]
#     SWEEP_UID = uids[0]

#     sweep_dir = Path("output/images/sweep")
#     sweep_dir.mkdir(parents=True, exist_ok=True)
#     config.IMAGES_DIR = sweep_dir

#     entry = t2i[SWEEP_UID]
#     ref_name = entry["reference_images"][0]["filename"]
#     ref_path = Path(IMAGES_DIR) / ref_name
#     prompt_text = entry["views"][0]["image_prompt"]
#     src_dims = (entry["source_dimensions"]["width"], entry["source_dimensions"]["height"])

#     results = []
#     for s in STRENGTHS:
#         config.SD3_IMG2IMG_STRENGTH = s
#         path = generate_image_sd3(
#             prompt_text, SWEEP_UID,
#             image_paths=[ref_path],
#             view_suffix=f"s{int(s * 100)}",
#             source_dimensions=src_dims,
#             mode="img2img",
#         )
#         results.append((s, path))

#     fig, axes = plt.subplots(1, len(results) + 1, figsize=(3.2 * (len(results) + 1), 4))
#     axes[0].imshow(Image.open(ref_path), cmap="gray")
#     axes[0].set_title("original", fontsize=10)
#     axes[0].axis("off")
#     for ax, (s, path) in zip(axes[1:], results):
#         ax.imshow(Image.open(path), cmap="gray")
#         ax.set_title(f"strength={s}", fontsize=10)
#         ax.axis("off")
#     plt.suptitle(f"img2img strength sweep, uid={SWEEP_UID}", y=1.02)
#     plt.tight_layout()
#     plt.show()

#     print("Low strength keeps the real X-ray almost intact; 1.0 ignores it entirely.")
#     print("Pick the lowest value that still reads as a distinct synthetic image.")

#     # Restore the default so later cells are unaffected.
#     config.SD3_IMG2IMG_STRENGTH = 0.75

## 15. Export — local zip download (disabled)

**Commented out** — this session relies on cell 16 (Kaggle Dataset publish)
as the sole export/backup path instead. Unlike 12b/13/14, this cell doesn't
strictly need img2img — it already skips `img2img` gracefully when there's
no metadata for it — so it's safe to uncomment on its own if a local zip is
ever wanted alongside the dataset publish.

Bundles originals and generated images per mode, with a CSV mapping, and zips
it for download. Only `output/` is packaged — the ~15 GB model cache in
`hf_cache/` is excluded, which also keeps the notebook commit from failing.

This is a one-off local copy for eyeballing results. For a persistent copy
that survives across sessions and can seed a future `--resume` run, see the
next cell instead.

In [ ]:
# import os, csv, json, shutil
# from pathlib import Path
# from IPython.display import FileLink, display

# EXPORT_DIR = Path("verification_data")
# if EXPORT_DIR.exists():
#     shutil.rmtree(EXPORT_DIR)

# rows = []
# for mode_name in ("txt2img", "img2img"):
#     meta_path = Path(f"output/metadata_{mode_name}.json")
#     if not meta_path.exists():
#         print(f"skipping {mode_name} — no metadata")
#         continue

#     synth_dir = EXPORT_DIR / mode_name / "synthetic"
#     orig_dir = EXPORT_DIR / mode_name / "original"
#     synth_dir.mkdir(parents=True, exist_ok=True)
#     orig_dir.mkdir(parents=True, exist_ok=True)

#     with open(meta_path) as f:
#         metadata = json.load(f)

#     for item in metadata:
#         uid = item.get("uid")
#         for view in item.get("views", []):
#             if "image_path" not in view:
#                 continue
#             src = Path(view["image_path"])
#             if not src.exists():
#                 continue
#             shutil.copy2(src, synth_dir / src.name)

#             orig_name = ""
#             for ref in item.get("reference_images", []):
#                 op = Path(IMAGES_DIR) / ref["filename"]
#                 if op.exists():
#                     shutil.copy2(op, orig_dir / op.name)
#                     orig_name = op.name
#                     break

#             params = view.get("sd3_params", {})
#             rows.append({
#                 "uid": uid,
#                 "requested_mode": mode_name,
#                 "effective_mode": params.get("mode", ""),
#                 "view": view.get("view") or "single",
#                 "synthetic": src.name,
#                 "original": orig_name,
#                 "steps": params.get("steps", ""),
#                 "guidance": params.get("guidance_scale", ""),
#                 "strength": params.get("strength", ""),
#                 "seed": params.get("seed", ""),
#                 "width": params.get("width", ""),
#                 "height": params.get("height", ""),
#             })
#     shutil.copy2(meta_path, EXPORT_DIR / mode_name / "metadata.json")

# if rows:
#     EXPORT_DIR.mkdir(parents=True, exist_ok=True)
#     with open(EXPORT_DIR / "mapping.csv", "w", newline="") as f:
#         writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
#         writer.writeheader()
#         writer.writerows(rows)

#     archive = shutil.make_archive("verification_data", "zip", EXPORT_DIR)
#     size_mb = os.path.getsize(archive) / 1024**2
#     print(f"{len(rows)} image(s) exported -> {archive} ({size_mb:.1f} MB)")

#     fell_back = [r for r in rows if r["requested_mode"] != r["effective_mode"]]
#     if fell_back:
#         print(f"\n{len(fell_back)} record(s) fell back to txt2img "
#               "(no reference image on disk) — see effective_mode in mapping.csv")

#     display(FileLink("verification_data.zip"))
# else:
#     print("Nothing to export — run cells 12a/12b first.")

## 16. Publish as a Kaggle Dataset (optional)

The durable checkpoint. Unlike the zip above, a published dataset survives
session resets and can be pulled back in by cell 7 at the start of a later
session — or attached as input to a different notebook entirely. Re-running
this cell adds a new version to the same dataset rather than duplicating it.

Requires the Kaggle API credentials from cell 5.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

# Keep the CLI current: an outdated kaggle-cli can return a bare 403 Forbidden
# for "dataset doesn't exist yet" instead of a clear not-found message, which
# defeats the create-fallback below if left unhandled (see cell 16 in
# Troubleshooting).
!pip install -q -U kaggle

# Must be unique to your account. Reuse the same slug across sessions so each
# publish adds a version instead of creating a duplicate dataset.
DATASET_SLUG = "synthetic-xrays-sd3"

if not HAS_KAGGLE_API:
    print("No Kaggle API credentials — see the 'Kaggle API credentials' cell (5).")
else:
    publish_dir = Path("kaggle_dataset_publish")
    if publish_dir.exists():
        shutil.rmtree(publish_dir)
    shutil.copytree("output", publish_dir / "output")

    full_slug = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
    metadata = {
        "title": "Synthetic Chest X-rays (SD3, Indiana reports)",
        "id": full_slug,
        "licenses": [{"name": "CC0-1.0"}],
    }
    (publish_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

    # "version" adds to an existing dataset; fall back to "create" the first
    # time, when the dataset doesn't exist yet. --dir-mode zip uploads the
    # whole tree as one archive instead of one request per PNG.
    #
    # The failure signal for "doesn't exist yet" is not consistent across
    # kaggle-cli versions — a clear "not found" on newer clients, a bare
    # "403 Forbidden" on older ones (permission is checked before existence).
    # So on ANY version failure, just try create: if the dataset turns out to
    # already exist, create's own "already exists" error means the original
    # version failure was a real problem (e.g. KAGGLE_USERNAME not matching
    # the dataset's owner) and gets surfaced instead.
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(publish_dir),
         "-m", "Checkpoint update", "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        version_error = (result.stdout + result.stderr).strip()
        print(f"`datasets version` failed, trying `datasets create` "
              f"(first publish, or the dataset doesn't exist yet):\n{version_error}\n")
        create_result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(publish_dir), "--dir-mode", "zip"],
            capture_output=True, text=True,
        )
        create_error = (create_result.stdout + create_result.stderr).lower()
        if create_result.returncode == 0:
            result = create_result
        elif "already exists" in create_error:
            print(
                "Dataset already exists, so `version` should have worked — the "
                "original error above is the real problem. Common cause: "
                f"KAGGLE_USERNAME ('{KAGGLE_USERNAME}') doesn't match the "
                "account that owns this dataset slug, or KAGGLE_KEY is stale "
                "(generate a fresh token at kaggle.com/settings)."
            )
            result = create_result
        else:
            result = create_result

    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"\nPublished to kaggle.com/datasets/{full_slug}")
        print(f'To resume from here in a future session, set DATASET_SLUG = '
              f'"{full_slug}" in cell 7 ("Resume from a previous Kaggle Dataset").')

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `403 Forbidden` / `GatedRepoError` on model load (cell 9) | The account behind `HF_TOKEN` hasn't accepted the SD3 Medium license yet — a valid token alone isn't enough | While logged into **that same account**, open [huggingface.co/stabilityai/stable-diffusion-3-medium](https://huggingface.co/stabilityai/stable-diffusion-3-medium) and click "Agree and access repository". Wait ~1 min, then re-run cell 9 |
| `FileNotFoundError` in "Check the dataset" (cell 8) | Older version of this notebook — this is fixed now; cell 8 never raises, it just disables img2img and aspect-ratio matching | Re-download/re-run this notebook; if it still raises, you have a stale copy |
| `NameError: name 'proj' is not defined` in "Prompt smoke test" | Cascade from the cell-8 crash above — `proj` never got created | Fixed alongside the above; `proj` (from the CSV, not the image files) is now always defined |
| `AttributeError: ... has no attribute 'enable_vae_slicing'` in cell 9/12a | The installed `diffusers` version doesn't expose that convenience method on `StableDiffusion3Pipeline` | Fixed in `sd3_generator.py` — it falls back to `pipe.vae.enable_slicing()`. Re-run cell 2 (`git pull`) to pick up the fix, then re-run cell 3's install |
| `CUDA out of memory` | SD3 Medium needs ~16-18 GB at fp16 | Cell 1 sets `SD3_ENABLE_CPU_OFFLOAD` automatically; if it still OOMs, set `SD3_DROP_T5=true` in `.env` (frees ~10 GB, but drops the long-prompt path) |
| OOM only during cells 12a/12b | The kernel still holds a copy while the subprocess loads its own | Re-run cell 9 — it releases the model at the end — or restart the kernel |
| `No space left on device` | The ~15 GB download went to the default cache | Cell 4 must run **before** any HF import; restart the session if it did not |
| Images are entirely black | bfloat16 on a pre-Ampere GPU (T4 = capability 7.5) | The loader falls back automatically; if you forced `SD3_DTYPE=bfloat16`, set `float16` |
| Text/labels rendered in the output | SD3 is unusually good at text; the negative prompt fights it | Raise `SD3_GUIDANCE_SCALE`, or add terms to `SD3_NEGATIVE_PROMPT` |
| img2img output looks identical to the original | `strength` too low | Raise it — below ~0.4 the result is a filtered copy, not a synthetic image |
| img2img produced txt2img output | No `--images-dir`, or the reference PNG is absent | Check `effective_mode` in `mapping.csv`, or the per-record warning the run prints |
| OpenAI `401` / `429` | Bad key, or rate limited | Check the `OPENAI_API_KEY` secret; 429 resolves on retry |
| Run stopped or the session was killed mid-batch | Kaggle's 12h cap, a manual stop, an OOM crash | Re-run the same cell (12a/12b) — it already passes `--resume`; metadata is saved after every image, so at most one is lost |
| Checkpoint status shows 0 complete right after a run that seemed to work | Reading the wrong `--output-subdir`, or a mismatched metadata filename | Cell 11 reads `output/metadata_<name>.json` directly — confirm the name matches the run's `--output-subdir` |
| `kaggle: command not found`, or dataset publish/download fails | Kaggle API credentials not set | Set `KAGGLE_USERNAME`/`KAGGLE_KEY` secrets and re-run cell 5 |
| `kaggle datasets version` says "not found" on the very first publish | Expected — cell 16 falls back to `datasets create` automatically |
| `403 Client Error: Forbidden` on `CreateDatasetVersion` (cell 16), often alongside an "outdated kaggle version" warning | The dataset doesn't exist yet, but an old `kaggle`-cli returns a bare 403 instead of "not found", so older notebook copies with a strict `"not found"` check never fell back to `create` — fixed now: cell 16 upgrades `kaggle` first and retries with `create` on **any** `version` failure | Re-download/re-run this notebook (`git pull`, cell 2) to pick up the fix. If `create` then reports "already exists", the real issue is `KAGGLE_USERNAME`/`KAGGLE_KEY` not matching the dataset's owner — refresh the token at kaggle.com/settings |
| `/kaggle/working` was empty after reopening the notebook | Kaggle does not guarantee working-directory persistence across sessions | Publish `output/` with cell 16 at the end of each session, pull it back with cell 7 at the start of the next |
| Notebook commit fails | The model cache is inside `/kaggle/working` | Only `output/` and the zip need to persist; delete `hf_cache/` before committing |
| Cell 12c says "Only 1 GPU visible" | Accelerator is set to a single GPU, or the T4 x2 setting didn't take effect | Session options (right sidebar) → Accelerator → GPU T4 x2 → save, then **restart the session** (a running session doesn't pick up an accelerator change) |
| Cell 12c: `torch.cuda.OutOfMemoryError` in a shard's log | Per-GPU VRAM exhausted — less common with offload on, but can happen from fragmentation or if `SD3_ENABLE_CPU_OFFLOAD` didn't take effect | Confirm `SD3_ENABLE_CPU_OFFLOAD=true` in `.env` (cell 1 sets it automatically per GPU's VRAM); if it's already on, set `DROP_T5 = True` in cell 6 and re-run cells 6 then 12c |
| Cell 12c: one shard's log shows a Python `MemoryError`, or the kernel just dies with no CUDA traceback at all | **Host RAM** exhausted, not VRAM — each shard CPU-offloads its own full model copy, roughly doubling RAM use vs. a single shard | Set `DROP_T5 = True` in cell 6 (frees ~9-10GB per shard — about half the model) and re-run cells 6 then 12c. If it still OOMs, fall back to cell 12a (one GPU, no doubling) |
| Cell 12c merge step reports fewer images than expected | One shard's process crashed partway (check its `.log` tail printed by the cell) | Re-run cell 12c — `--resume` means only the missing uids from the crashed shard are regenerated |

**Costs and timing.** Prompt extraction with `gpt-4.1-mini` is roughly $0.65 for
all ~1000 reports. Generation is the real budget: ~8-15 s/image on an L4,
~60-120 s/image on a T4 with offload. Kaggle allows 30 GPU-hours/week and 12
hours per session, so a full 1000-image run is an L4 job, likely spanning
several sessions — which is exactly what `--resume` and the dataset checkpoint
cells are for.